In [1]:
from pathlib import Path

import pandas as pd

In [2]:
data = Path().resolve() / "metadata"

In [3]:
node_table = pd.read_csv(data.parent / "network_analysis/network_node_table.csv")

In [4]:
node_table = node_table.rename(columns={
    'group_mash' : 'mash_cluster',
    'group_pling' : 'pling_subcommunity'
})

In [5]:
metadata = pd.read_csv(data.parent / 'ncbi_metadata.csv', sep=';')
contig_info = pd.read_csv(data.parent / 'contig_info.csv')
resfinder = pd.read_csv(data.parent / 'resfinder/resfinder_results.csv')
mobtyper = pd.read_csv(data.parent / 'mobtyper/mobtyper_merged.csv')

In [6]:
df = (
    node_table
    .merge(
        metadata,
        on='accession',
        how='left'
    )
    .merge(
        contig_info,
        on='accession',
        how='left'
    )
    .merge(
        resfinder,
        left_on='accession',
        right_on='SEQUENCE',
        how='left'
    )
    .drop(columns='SEQUENCE')
    .merge(
        mobtyper,
        left_on='accession',
        right_on='sample_id',
        how='left'
    )
    .drop(columns='sample_id')
)

In [7]:
df = df[[col for col in df.columns if not col.endswith('_y')]]
df.columns = [col.replace("_x", "") for col in df.columns]

In [8]:
df.to_csv(data / "metadata_merged.csv", index=False)

In [9]:
res_long = df.copy()
res_long['resfinder_profile'] = res_long['resfinder_profile'].str.split(', ')
res_long = res_long.explode('resfinder_profile')
res_long['resfinder_profile'] = res_long['resfinder_profile'].replace('none', pd.NA)
res_long = res_long.drop_duplicates()

In [10]:
mash_resfinder = (
    res_long
    .groupby('mash_cluster')
    ['resfinder_profile']
    .value_counts()
    .reset_index(name='gene_count')
    .merge(
        df['mash_cluster']
        .value_counts()
        .reset_index(),
        on='mash_cluster')
    .set_index('mash_cluster')
    .apply(lambda x: f'{x['resfinder_profile']} ({x['gene_count'] / x['count']:.0%}, {x['gene_count']}/{x['count']})', axis=1)
    .groupby(level=0)
    .apply(lambda s: '; '.join(s))
    .reset_index(name='resfinder_profile')
)

In [11]:
pling_resfinder = ( 
    res_long
    .groupby('pling_subcommunity')
    ['resfinder_profile']
    .value_counts()
    .reset_index(name='gene_count')
    .merge(
        df['pling_subcommunity']
        .value_counts()
        .reset_index(),
        on='pling_subcommunity')
    .set_index('pling_subcommunity')
    .apply(lambda x: f'{x['resfinder_profile']} ({x['gene_count'] / x['count']:.0%}, {x['gene_count']}/{x['count']})', axis=1)
    .groupby(level=0)
    .apply(lambda s: '; '.join(s))
    .reset_index(name='resfinder_profile')
)

In [12]:
rep_long = df.copy()
rep_long['rep_type(s)'] = rep_long['rep_type(s)'].str.split(',')
rep_long = rep_long.explode('rep_type(s)')
rep_long['rep_type(s)'] = rep_long['rep_type(s)'].replace('-', pd.NA)
rep_long = rep_long.drop_duplicates()

In [13]:
mash_reps = (    rep_long
    .groupby('mash_cluster')
    ['rep_type(s)']
    .value_counts()
    .reset_index(name='gene_count')
    .merge(
        df['mash_cluster']
        .value_counts()
        .reset_index(),
        on='mash_cluster')
    .set_index('mash_cluster')
    .apply(lambda x: f'{x['rep_type(s)']} ({x['gene_count'] / x['count']:.0%}, {x['gene_count']}/{x['count']})', axis=1)
    .groupby(level=0)
    .apply(lambda s: '; '.join(s))
    .reset_index(name='rep_type(s)')
)

In [14]:
pling_reps = (    rep_long
    .groupby('pling_subcommunity')
    ['rep_type(s)']
    .value_counts()
    .reset_index(name='gene_count')
    .merge(
        df['pling_subcommunity']
        .value_counts()
        .reset_index(),
        on='pling_subcommunity')
    .set_index('pling_subcommunity')
    .apply(lambda x: f'{x['rep_type(s)']} ({x['gene_count'] / x['count']:.0%}, {x['gene_count']}/{x['count']})', axis=1)
    .groupby(level=0)
    .apply(lambda s: '; '.join(s))
    .reset_index(name='rep_type(s)')
)

In [15]:
rep_long = df.copy()
rep_long['relaxase_type(s)'] = rep_long['relaxase_type(s)'].str.split(',')
rep_long = rep_long.explode('relaxase_type(s)')
rep_long['relaxase_type(s)'] = rep_long['relaxase_type(s)'].replace('-', pd.NA)
rep_long = rep_long.drop_duplicates()

In [16]:
mash_relax= (    rep_long
    .groupby('mash_cluster')
    ['relaxase_type(s)']
    .value_counts()
    .reset_index(name='gene_count')
    .merge(
        df['mash_cluster']
        .value_counts()
        .reset_index(),
        on='mash_cluster')
    .set_index('mash_cluster')
    .apply(lambda x: f'{x['relaxase_type(s)']} ({x['gene_count'] / x['count']:.0%}, {x['gene_count']}/{x['count']})', axis=1)
    .groupby(level=0)
    .apply(lambda s: '; '.join(s))
    .reset_index(name='relaxase_type(s)')
)

In [17]:
pling_relax= (    rep_long
    .groupby('pling_subcommunity')
    ['relaxase_type(s)']
    .value_counts()
    .reset_index(name='gene_count')
    .merge(
        df['pling_subcommunity']
        .value_counts()
        .reset_index(),
        on='pling_subcommunity')
    .set_index('pling_subcommunity')
    .apply(lambda x: f'{x['relaxase_type(s)']} ({x['gene_count'] / x['count']:.0%}, {x['gene_count']}/{x['count']})', axis=1)
    .groupby(level=0)
    .apply(lambda s: '; '.join(s))
    .reset_index(name='relaxase_type(s)')
)

In [18]:
rep_long = df.copy()
rep_long['mpf_type'] = rep_long['mpf_type'].str.split(',')
rep_long = rep_long.explode('mpf_type')
rep_long['mpf_type'] = rep_long['mpf_type'].replace('-', pd.NA)
rep_long = rep_long.drop_duplicates()

In [19]:
mash_mpf = (    rep_long
    .groupby('mash_cluster')
    ['mpf_type']
    .value_counts()
    .reset_index(name='gene_count')
    .merge(
        df['mash_cluster']
        .value_counts()
        .reset_index(),
        on='mash_cluster')
    .set_index('mash_cluster')
    .apply(lambda x: f'{x['mpf_type']} ({x['gene_count'] / x['count']:.0%}, {x['gene_count']}/{x['count']})', axis=1)
    .groupby(level=0)
    .apply(lambda s: '; '.join(s))
    .reset_index(name='mpf_type')
)

In [20]:
pling_mpf= (    rep_long
    .groupby('pling_subcommunity')
    ['mpf_type']
    .value_counts()
    .reset_index(name='gene_count')
    .merge(
        df['pling_subcommunity']
        .value_counts()
        .reset_index(),
        on='pling_subcommunity')
    .set_index('pling_subcommunity')
    .apply(lambda x: f'{x['mpf_type']} ({x['gene_count'] / x['count']:.0%}, {x['gene_count']}/{x['count']})', axis=1)
    .groupby(level=0)
    .apply(lambda s: '; '.join(s))
    .reset_index(name='mpf_type')
)

In [21]:
def median_iqr(x):
    return f'{x.median():.2f} [{x.quantile(.25):.2f}-{x.quantile(.75):.2f}]'

In [22]:
def count_agg(x):
    total = x.shape[0]
    x = x.fillna("None")
    counts = x.value_counts()
    counts = counts.apply(lambda x: f'{x / total:.0%} ({x}/{total})')
    return '; '.join([f'{a} {b}' for a, b in zip(counts.index, counts.values)])

In [23]:
(
    df
    .groupby('mash_cluster')
    .agg({
        'size' : median_iqr,
        'gc' : median_iqr,
        'cfr_gene' : count_agg,
        'genus': count_agg,
        'predicted_mobility' : count_agg
        })
    .reset_index()
    .rename(columns={
        'size' : 'bp (median[IQR])',
        'gc' : 'gc (median[IQR])'
    })
    .merge(
        mash_resfinder,
        on='mash_cluster',
        how='left'
    )
    .merge(
        mash_reps,
        on='mash_cluster',
        how='left'
    )
    .merge(
        mash_relax,
        on='mash_cluster',
        how='left'
    )
    .merge(
        mash_mpf,
        on='mash_cluster',
        how='left'
    )
    .fillna('-')
).to_csv(data / 'mash_cluster_description.csv', index=False)

In [24]:
(
    df
    .groupby('pling_subcommunity')
    .agg({
        'size' : median_iqr,
        'gc' : median_iqr,
        'cfr_gene' : count_agg,
        'genus': count_agg,
        'predicted_mobility' : count_agg
        })
    .reset_index()
    .rename(columns={
        'size' : 'bp (median[IQR])',
        'gc' : 'gc (median[IQR])'
    })
    .merge(
        pling_resfinder,
        on='pling_subcommunity',
        how='left'
    )
    .merge(
        pling_reps,
        on='pling_subcommunity',
        how='left'
    )
    .merge(
        pling_relax,
        on='pling_subcommunity',
        how='left'
    )
    .merge(
        pling_mpf,
        on='pling_subcommunity',
        how='left'
    )
    .fillna('-')
).to_csv(data / 'pling_subcommunity_description.csv', index=False)